In [ ]:
import sys
import warnings
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from tabpfn import TabPFNRegressor

sys.path.append("..")
from src.data_generation.data_preperation import grid_from_cfg
from src.data_generation.noise import (
    quote_data_preparation, make_quote_eval_set, noisy_data_preparation,
)
from src.model.finetune import finetune
from src.model.quote_loss import quote_arb_loss
from src.model.SSVI import fit_ssvi, predict_ssvi
from src.evaluation.surface_eval import eval_surfaces, check_arbitrage_flat, inside_spread_fraction

cfg = yaml.safe_load(open("../config.yaml"))
ttms, ks = grid_from_cfg(cfg)
GRID_SHAPE = (len(ttms), len(ks))

warnings.filterwarnings("ignore", message="Running on CPU with more than")

In [ ]:
N_HELDOUT = 15
RUN_NAME = "ssvi_quote_uniform_3_60"

data_provider = partial(quote_data_preparation, cfg, n_context=(3, 60), n_heldout=N_HELDOUT)
loss_fn = partial(quote_arb_loss, grid_shape=GRID_SHAPE, lambda_cal=1.0, lambda_bf=1.0)
val_sets = [make_quote_eval_set(cfg, 8, s, N_HELDOUT) for s in (5, 10, 20, 40, 60)]
val_data = (sum((v[0] for v in val_sets), []), sum((v[1] for v in val_sets), []))

finetune(data_provider, run_name=RUN_NAME, n_epochs=300, n_surfaces_per_epoch=200,
         batch_size=4, val_every=5, val_data=val_data, loss_fn=loss_fn)

In [ ]:
N_ESTIMATORS = 1

def load_finetuned(run_name, which="final"):
    model = TabPFNRegressor(
        fit_mode="fit_preprocessors", n_estimators=N_ESTIMATORS,
        inference_config={"FINGERPRINT_FEATURE": False},
    )
    model._initialize_model_variables()
    state = torch.load(f"../checkpoints/{run_name}/{which}.pt", map_location="cpu")
    model.model_.load_state_dict(state)
    return model, state

baseline = TabPFNRegressor(n_estimators=N_ESTIMATORS, inference_config={"FINGERPRINT_FEATURE": False})
quote_ft, quote_state = load_finetuned(RUN_NAME)
noisy_ft, noisy_state = load_finetuned("ssvi_noisy_uniform_3_60")  # supervised ceiling

In [ ]:
def split_quotes(train):
    out = []
    for X, y in train:
        n = len(y) // 2
        out.append((X[:n, :2], (y[:n] + y[n:]) / 2, (y[n:] - y[:n]) / 2))
    return out


def refit_wls(train, test):
    KK = ks[None, :] * np.sqrt(ttms[:, None])          # physical strike wedge k = z·√τ
    maes = []
    for (X2, mid, s), (Xq, yq) in zip(split_quotes(train), test):
        w = 1 / np.maximum(2 * mid * X2[:, 1] * s, 1e-10)
        params, _ = fit_ssvi(X2, mid, cfg, weights=w)
        maes.append(np.mean(np.abs(predict_ssvi(params, ttms, KK).ravel() - yq)))
    return np.mean(maes)


def mid_strawman(train, test):
    errs = []
    for (X, y), (Xq, yq) in zip(train, test):
        n = len(y) // 2
        mid = (y[:n] + y[n:]) / 2
        idx = [np.where((Xq[:, 0] == X[i, 0]) & (Xq[:, 1] == X[i, 1]))[0][0] for i in range(n)]
        errs.append(np.abs(mid - yq[idx]))
    return np.concatenate(errs).mean()

In [ ]:
N_TEST = 50

for m in (0.5, 1.0, 2.0):
    print(f"\n=== m={m} (MAE vs truth) ===")
    print(f"{'n_ctx':>6} {'FT (arb)':>10} {'FT (supervised)':>15} {'SSVI refit (WLS)':>17} {'mid MAE':>8}")
    for n_ctx in (3, 5, 10, 20, 40, 60):
        tr, te = noisy_data_preparation(cfg, N_TEST, n_ctx, regime=m)
        q = eval_surfaces(quote_ft, tr, te, cfg, reload_state=quote_state)
        f = eval_surfaces(noisy_ft, tr, te, cfg, reload_state=noisy_state)
        r = refit_wls(tr, te)
        s = mid_strawman(tr, te)
        print(f"{n_ctx:>6} {q[0]:>10.4f} {f[0]:>15.4f} {r:>17.4f} {s:>8.4f}")

In [ ]:
for n_ctx in (5, 10, 20, 40, 60):
    tr, te = noisy_data_preparation(cfg, 25, n_ctx, regime=1.0)
    cal_v, bf_v = [], []
    for (Xn, yn), (Xq, yq) in zip(tr, te):
        quote_ft.fit(Xn, yn)
        quote_ft.model_.load_state_dict(quote_state)
        c, b = check_arbitrage_flat(cfg, quote_ft.predict(Xq))
        cal_v.append(c); bf_v.append(b)
    frac = inside_spread_fraction(quote_ft, tr, reload_state=quote_state)
    print(f"n_ctx={n_ctx}: cal_arb={np.mean(cal_v):.1%} butterfly_arb={np.mean(bf_v):.1%} inside_spread={frac:.1%}")

In [ ]:
tr, te = noisy_data_preparation(cfg, 1, 10, regime=2.0)
(Xn, yn), (Xq, yq) = tr[0], te[0]
quote_ft.fit(Xn, yn)
quote_ft.model_.load_state_dict(quote_state)
pred = quote_ft.predict(Xq)

n = len(yn) // 2
kq, tq, bid, ask = Xn[:n, 0], Xn[:n, 1], yn[:n], yn[n:]
zq = kq / np.sqrt(tq)                                  # quotes back to z = k/√τ for a shared x-axis
fig, axes = plt.subplots(5, 3, figsize=(15, 18), sharex=True, sharey=True)
for t_i, ax in enumerate(axes.ravel()):
    sel = slice(t_i * len(ks), (t_i + 1) * len(ks))
    ax.plot(ks, yq[sel], "k-", label="true (never trained on)")
    ax.plot(ks, pred[sel], "C0--", label="prediction")
    q = np.isclose(tq, ttms[t_i])
    if q.any():
        ax.errorbar(zq[q], (bid[q] + ask[q]) / 2, yerr=(ask[q] - bid[q]) / 2,
                    fmt="C3.", capsize=3, label="quotes (bid/ask)")
    ax.set_title(f"tau={ttms[t_i]:.3f}  ({int(q.sum())} quotes)", fontsize=10)
for ax in axes[-1]:
    ax.set_xlabel("z = k/√τ")
for ax in axes[:, 0]:
    ax.set_ylabel("IV")
axes[0, 0].legend()
plt.tight_layout()
plt.show()